# Game Script — Locked Design and EDA 4 Planning
*Recorded May 18, 2026 — pre-EDA 2 coding session*

---

## Background and Motivation

Game script features are derived from `raw.plays` using the score differential at the time
of each play. They capture how teams behave and perform under different competitive
pressures — information that season-level aggregates obscure by mixing together plays
from blowouts, close games, and deficit situations.

**Two distinct layers:**

- **Distribution layer** — how much time does each team spend in each game script bucket.
  This is a team identity characteristic independent of how they perform within each bucket.
- **Performance layer** — EPA per play, rush rate within each bucket. Measures execution
  quality conditional on the situation.

**Key design principle:** College football thresholds are wider than NFL equivalents.
Scoring is lower on average, games stay closer longer, and the talent gap between
conference opponents is wider. Thresholds calibrated accordingly.

---

## Bucket Definitions

| Bucket | Score differential at time of play |
|---|---|
| Neutral | Within ±10 |
| Ahead | Leading 11–20 |
| Blowout lead | Leading 21+ |
| Behind | Trailing 11–20 |
| Blowout deficit | Trailing 21+ |

**Garbage time filter:** Leading 28+ in Q4. Plays meeting this definition are excluded
from all bucket calculations. Not a feature bucket — a filter only.

Score differential is defined from the perspective of the team whose features are being
computed. Positive = leading, negative = trailing.

---

## Metrics Computed Per Bucket

| Metric | Neutral | Ahead | Blowout lead | Behind | Blowout deficit |
|---|---|---|---|---|---|
| Play count | ✓ | ✓ | ✓ | ✓ | ✓ |
| EPA per play (off) | ✓ | ✓ | — | ✓ | ✓ |
| EPA per play (def) | ✓ | ✓ | — | ✓ | ✓ |
| Rush rate | ✓ | ✓ | ✓ | ✓ | ✓ |
| Pct of total plays | ✓ | ✓ | ✓ | ✓ | ✓ |

EPA excluded from blowout lead — too contaminated by personnel and motivation decisions
to be a clean signal. Rush rate retained because it captures game management and personnel
decisions that are themselves informative.

---

## Why Garbage Time Matters for Total Points

Early blowouts inflate total points through garbage time scoring against depleted defenses.
A game ending 59-21 where the blowout was established by halftime is not the same scoring
environment as a game ending 59-21 because both offenses were genuinely explosive for four
quarters. The garbage time filter (leading 28+ in Q4) excludes plays from the period where
winning teams deploy backups and losing teams accumulate low-value passing yards against
prevent coverage. This is the primary mechanism by which game script features become
contaminated for total points prediction.

---

## Season-Level Feature

**`gs_neutral_pct_volatility`** — standard deviation of neutral-play percentage across
games within a season. One value per team-season. Written to `int_team_season_features`.

Captures whether a team consistently plays in contested games or swings between blowout
and deficit situations across their schedule. Strong prior for the total model — teams
involved in high-variance scoring environments have systematically different total points
distributions than teams that grind out close games every week.

---

## CTD Construction

All per-game metrics aggregated to cumulative averages through the prior game in the
season. Game N uses the average of Games 1 through N-1. Identical CTD pattern used
for all other rolling features in this build.

Week 5 is the first game in the prediction window. Teams will have 1–4 prior games
of CTD data at that point depending on bye weeks and schedule. Expected and consistent
with all other CTD features — the prior carries more weight when CTD sample is thin,
which is the intended Bayesian behavior.

---

## Schema Destination

### `int_game_team_features` — CTD columns (gs_ prefix)

```
gs_neutral_plays_ctd
gs_neutral_off_epa_ctd
gs_neutral_def_epa_ctd
gs_neutral_rush_rate_ctd
gs_neutral_pct_ctd

gs_ahead_plays_ctd
gs_ahead_off_epa_ctd
gs_ahead_def_epa_ctd
gs_ahead_rush_rate_ctd
gs_ahead_pct_ctd

gs_blowout_lead_plays_ctd
gs_blowout_lead_rush_rate_ctd
gs_blowout_lead_pct_ctd

gs_behind_plays_ctd
gs_behind_off_epa_ctd
gs_behind_def_epa_ctd
gs_behind_rush_rate_ctd
gs_behind_pct_ctd

gs_blowout_deficit_plays_ctd
gs_blowout_deficit_off_epa_ctd
gs_blowout_deficit_def_epa_ctd
gs_blowout_deficit_rush_rate_ctd
gs_blowout_deficit_pct_ctd
```

### `int_team_season_features` — season-level

```
gs_neutral_pct_volatility
```

---

## Candidate Feature Flags for candidate_features.csv

| Feature group | Prior strength | Primary target | Notes |
|---|---|---|---|
| Neutral-script EPA (off + def) | Standard | Spread + total | Cleanest game script signal. Total prior moderate due to scoring environment correlation. |
| Neutral rush rate | Standard | Spread + total | Playcalling identity. Cleaner for total than EPA-based features. |
| Neutral pct | Standard | Total primarily | Distribution layer. Clean causal story for total. |
| Ahead/behind EPA | Standard | Spread primarily | Execution under competitive pressure. Weaker total prior. |
| Ahead/behind rush rate | Standard | Spread primarily | Play calling adjustment. |
| Blowout lead/deficit rush rate | Standard | Spread primarily | Personnel and game management signal. |
| Blowout lead/deficit pct | Standard | Total primarily | How often team is in non-competitive territory. |
| Distribution pcts (all buckets) | Standard | Total primarily | Cleanest game script signal for total model. |
| gs_neutral_pct_volatility | Standard | Total primarily | Schedule variance proxy. Strong total prior. |
| Blowout lead/deficit EPA | Low prior | Spread only | Excluded from blowout lead. Contaminated by personnel in deficit. |

---

## What Is Explicitly Deferred

- **Clustering on game script distribution** — deferred to post-EDA 4. If distribution
  percentage features show strong signal, clustering becomes a motivated decision backed
  by evidence. Building it now is speculative architecture.
- **Delta features between buckets** — e.g. neutral EPA minus behind EPA as a measure
  of execution drop-off under pressure. Deferred to EDA 4. Build only if bucket-level
  features show signal worth refining.
- **Interaction terms between script buckets and style archetypes** — deferred to EDA Final.
- **Any total-model-specific game script features beyond what is listed** — EDA 4 decides.

---

## EDA 4 — Game Script Testing Plan

*These are the questions EDA 4 must answer for game script features specifically.
EDA 4 structure is not fully planned until EDA 2 inventory is complete — this section
records the game script questions so they are not lost.*

### Core signal questions

**1. Does neutral-script EPA outperform overall EPA?**
Test partial correlation of `gs_neutral_off_epa_ctd` vs `off_epa_per_play` (overall)
against point differential and total points separately. If neutral-script EPA adds
no signal over overall EPA, the additional construction complexity is not justified.
This is the most important test for the entire game script feature family.

**2. Do the bucket-specific EPAs have independent signal beyond neutral?**
After controlling for neutral-script EPA, do behind-script EPA or ahead-script EPA
add predictive value for spread? This tests whether execution under specific pressures
captures something beyond general quality.

**3. Do distribution percentages predict total points independently?**
Test `gs_neutral_pct_ctd`, `gs_blowout_lead_pct_ctd`, `gs_blowout_deficit_pct_ctd`
against total points. The hypothesis is that teams involved in more blowouts (either
direction) have systematically different total points distributions. This is the
cleanest game script signal for the total model.

**4. Does `gs_neutral_pct_volatility` predict total points?**
Teams with high volatility in how contested their games are should have wider total
points distributions. Test as a direct predictor and as a moderating variable.

**5. Is there a season segment interaction?**
Game script features derived from early-season games (weeks 1-4, pre-prediction window)
may be less stable than those derived from mid-season conference play. Test whether
CTD game script features have stronger signal in late-season weeks than early-season
weeks, consistent with the EDA 3 season segment boundary findings.

### Garbage time validation

**6. Does excluding garbage time plays change the signal?**
Compare feature values computed with and without the garbage time filter for a sample
of games. Verify that the filter meaningfully changes values for teams with frequent
blowouts and has minimal effect for teams that play close games throughout. This
validates that the filter is doing real work and not just reducing sample size.

### Clustering decision gate

**7. If distribution percentage features show signal — does clustering add anything?**
Only reached if Question 3 returns positive. Run KMeans on the five distribution
percentage features (pct of plays in each bucket). Test whether cluster membership
has predictive signal beyond the raw percentages. If yes, clusters enter the candidate
list. If no, raw percentages are sufficient and clustering is not pursued.

### Decision outputs EDA 4 must produce for game script

For each feature tested, the standard EDA 4 output applies:

- Signal vs point differential: partial r, conference scope, season segment breakdown
- Signal vs total points: partial r, conference scope, season segment breakdown
- YoY stability: stable enough to be a prior seed, a game-level predictor, or neither
- Verdict: include / exclude / conditional, with full justification from output

Additionally for game script specifically:
- Whether neutral-script features replace or supplement overall EPA features
- Whether the behind/ahead buckets add signal worth the schema complexity
- Whether clustering on distribution is warranted
- Whether any delta features (e.g. neutral minus behind EPA) should be built before EDA Final

---

## Implementation Note for EDA 2 Coding Session

Build Phase 1 only. No clustering, no delta features, no interaction terms.

**Source:** `raw.plays` — score differential at time of play, EPA, play type (run/pass),
down and distance for down-type classification.

**Join path:** `raw.plays` → `raw.games` for game metadata → `int_team_season_features`
for conference filters. Both teams must pass full rebuild pool filter.

**Output:** 24 CTD columns in `int_game_team_features` + 1 season-level column in
`int_team_season_features`. Full rebuild pool filter applied. All assertions scoped
to rebuild pool (3,458 team-rows), never against full table.

In [50]:
# CELL 1 — DIAGNOSTIC — Full Column Inventory and Baseline Audit

import psycopg2
import pandas as pd
import numpy as np

# ── Global Constants ──────────────────────────────────────────────
RANDOM_SEED = 42
TRAIN_SEASONS = [2022, 2023, 2024, 2025]
TEST_SIZE = 0.20
CONFERENCE_GAMES_ONLY = True
EXCLUDE_INDEPENDENTS = True

conn = psycopg2.connect(
    host='127.0.0.1', port=5455, dbname='postgres',
    user='postgres', password='postgres'
)
cur = conn.cursor()

print(f"RANDOM_SEED={RANDOM_SEED} | TRAIN_SEASONS={TRAIN_SEASONS} | "
      f"TEST_SIZE={TEST_SIZE} | CONFERENCE_GAMES_ONLY={CONFERENCE_GAMES_ONLY} | "
      f"EXCLUDE_INDEPENDENTS={EXCLUDE_INDEPENDENTS}")

# ── Full column inventory — all int tables ────────────────────────
cur.execute("""
    SELECT table_name, column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'int'
    ORDER BY table_name, ordinal_position
""")
rows = cur.fetchall()
inv = pd.DataFrame(rows, columns=['table', 'column', 'type', 'nullable'])

print("\n=== INT SCHEMA — FULL COLUMN INVENTORY ===")
for table in inv['table'].unique():
    t = inv[inv['table'] == table]
    print(f"\n--- {table} ({len(t)} columns) ---")
    for _, r in t.iterrows():
        print(f"  {r['column']:<50} {r['type']:<25} nullable={r['nullable']}")

# ── Row counts per int table ──────────────────────────────────────
print("\n=== INT TABLE ROW COUNTS ===")
for table in inv['table'].unique():
    cur.execute(f"SELECT COUNT(*) FROM int.{table}")
    n = cur.fetchone()[0]
    print(f"  {table}: {n:,}")

# ── raw.plays column inventory ────────────────────────────────────
cur.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'raw' AND table_name = 'plays'
    ORDER BY ordinal_position
""")
plays_cols = cur.fetchall()
print("\n=== raw.plays — FULL COLUMN INVENTORY ===")
for col, dtype in plays_cols:
    print(f"  {col:<50} {dtype}")

cur.execute("SELECT COUNT(*) FROM raw.plays")
print(f"\nraw.plays total rows: {cur.fetchone()[0]:,}")

RANDOM_SEED=42 | TRAIN_SEASONS=[2022, 2023, 2024, 2025] | TEST_SIZE=0.2 | CONFERENCE_GAMES_ONLY=True | EXCLUDE_INDEPENDENTS=True

=== INT SCHEMA — FULL COLUMN INVENTORY ===

--- int_game_environment (22 columns) ---
  game_id                                            bigint                    nullable=YES
  season                                             integer                   nullable=YES
  week                                               integer                   nullable=YES
  home_team                                          text                      nullable=YES
  away_team                                          text                      nullable=YES
  venue_id                                           bigint                    nullable=YES
  is_dome                                            boolean                   nullable=YES
  venue_elevation_ft                                 double precision          nullable=YES
  away_home_elevation_ft                        

In [51]:
# CELL 2 — DIAGNOSTIC — Identify what sacks column represents

cur.execute("""
    SELECT
        tsf.team_name,
        tsf.season,
        tsf.sacks,
        tsf.tackles_for_loss,
        tsf.pass_attempts,
        tsf.def_havoc_total,
        tsf.def_havoc_front_seven
    FROM int.int_team_season_features tsf
    WHERE tsf.season IN (2023, 2024)
      AND tsf.team_name IN (
          'Alabama', 'Georgia', 'Michigan', 'Ohio State',
          'Texas', 'Penn State', 'Ole Miss', 'Iowa',
          'Air Force', 'Kennesaw State'
      )
    ORDER BY tsf.season, tsf.sacks DESC
""")
rows = cur.fetchall()
cols = ['team_name','season','sacks','tackles_for_loss',
        'pass_attempts','def_havoc_total','def_havoc_front_seven']
df = pd.DataFrame(rows, columns=cols)
for c in ['sacks','tackles_for_loss','pass_attempts','def_havoc_total','def_havoc_front_seven']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

print("=== Sack column diagnostic ===")
print(df.to_string(index=False))

# ── Range check across full training pool ─────────────────────────
cur.execute("""
    SELECT
        MIN(sacks)  AS min_sacks,
        MAX(sacks)  AS max_sacks,
        AVG(sacks)  AS avg_sacks,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY sacks) AS p25,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY sacks) AS p50,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY sacks) AS p75
    FROM int.int_team_season_features
    WHERE season IN (2022, 2023, 2024, 2025)
      AND conference != 'FBS Independents'
      AND conference != 'Pac-12'
      AND NOT (conference = 'American Athletic' AND season = 2022)
      AND NOT (conference = 'Mountain West'     AND season = 2022)
      AND NOT (conference = 'Big 12'            AND season = 2022)
""")
row = cur.fetchone()
print(f"\n=== sacks distribution (training pool) ===")
print(f"  min={row[0]}  p25={float(row[3]):.1f}  p50={float(row[4]):.1f}  "
      f"p75={float(row[5]):.1f}  max={row[1]}  avg={float(row[2]):.1f}")

# ── Also pull raw.plays sack play_type values to see what exists ──
cur.execute("""
    SELECT play_type, COUNT(*) AS n
    FROM raw.plays
    WHERE LOWER(play_type) LIKE '%sack%'
    GROUP BY play_type
    ORDER BY n DESC
""")
sack_types = cur.fetchall()
print(f"\n=== raw.plays — play_type values containing 'sack' ===")
for pt, n in sack_types:
    print(f"  {pt:<40} {n:,}")

=== Sack column diagnostic ===
     team_name  season  sacks  tackles_for_loss  pass_attempts  def_havoc_total  def_havoc_front_seven
    Penn State    2023     50               112            412            0.225                  0.171
       Alabama    2023     40                84            325            0.185                  0.116
      Michigan    2023     38                84            361            0.215                  0.132
      Ole Miss    2023     35                82            395            0.175                  0.108
          Iowa    2023     32                75            348            0.157                  0.094
     Air Force    2023     30                68            105            0.158                  0.105
         Texas    2023     29                79            475            0.180                  0.108
       Georgia    2023     29                72            456            0.171                  0.097
    Ohio State    2023     28             

In [52]:
# CELL 3 — DIAGNOSTIC — raw.plays play_type distribution

cur.execute("""
    SELECT play_type, COUNT(*) AS n
    FROM raw.plays
    WHERE season IN (2022, 2023, 2024, 2025)
    GROUP BY play_type
    ORDER BY n DESC
""")
rows = cur.fetchall()
print("=== raw.plays play_type distribution (2022-2025) ===")
for pt, n in rows:
    print(f"  {pt:<50} {n:,}")

=== raw.plays play_type distribution (2022-2025) ===
  Rush                                               387,967
  Pass Reception                                     203,532
  Pass Incompletion                                  137,896
  Penalty                                            54,189
  Punt                                               53,117
  Kickoff                                            48,142
  Timeout                                            45,104
  Sack                                               22,486
  Rushing Touchdown                                  19,207
  Passing Touchdown                                  18,885
  Kickoff Return (Offense)                           17,327
  End Period                                         13,877
  Field Goal Good                                    13,025
  Fumble Recovery (Own)                              5,287
  Interception                                       4,782
  End of Half                                 

In [53]:
# CELL 4 — DIAGNOSTIC — TFL back-calculation check

cur.execute("""
    SELECT
        p.season,
        p.offense,
        COUNT(*) FILTER (WHERE p.play_type = 'Sack')                          AS sacks_from_plays,
        COUNT(*) FILTER (WHERE p.play_type = 'Rush' AND p.yards_gained < 0)   AS neg_rush_plays,
        COUNT(*) FILTER (WHERE p.play_type = 'Sack'
                            OR (p.play_type = 'Rush' AND p.yards_gained < 0)) AS approx_tfl
    FROM raw.plays p
    WHERE p.season IN (2023, 2024)
      AND p.offense IN (
          'Alabama', 'Georgia', 'Michigan', 'Ohio State',
          'Texas', 'Penn State', 'Ole Miss', 'Iowa'
      )
    GROUP BY p.season, p.offense
    ORDER BY p.season, p.offense
""")
rows = cur.fetchall()
play_df = pd.DataFrame(rows, columns=['season','team','sacks_plays','neg_rush','approx_tfl'])
play_df[['sacks_plays','neg_rush','approx_tfl']] = play_df[['sacks_plays','neg_rush','approx_tfl']].astype(float)

cur.execute("""
    SELECT team_name, season, tackles_for_loss, sacks
    FROM int.int_team_season_features
    WHERE season IN (2023, 2024)
      AND team_name IN (
          'Alabama', 'Georgia', 'Michigan', 'Ohio State',
          'Texas', 'Penn State', 'Ole Miss', 'Iowa'
      )
    ORDER BY season, team_name
""")
tsf_rows = cur.fetchall()
tsf_df = pd.DataFrame(tsf_rows, columns=['team','season','tfl_actual','sacks_actual'])
tsf_df[['tfl_actual','sacks_actual']] = tsf_df[['tfl_actual','sacks_actual']].astype(float)

merged = play_df.merge(tsf_df, on=['team','season'])
merged['tfl_gap'] = merged['tfl_actual'] - merged['approx_tfl']
merged['pct_captured'] = (merged['approx_tfl'] / merged['tfl_actual'] * 100).round(1)

print("=== TFL back-calculation vs actual ===")
print(f"{'Team':<15} {'Seas':>4} {'Sacks(plays)':>13} {'NegRush':>8} "
      f"{'ApproxTFL':>10} {'ActualTFL':>10} {'Gap':>6} {'%Captured':>10}")
print("-" * 80)
for _, r in merged.iterrows():
    print(f"  {r['team']:<13} {int(r['season']):>4} {r['sacks_plays']:>13.0f} "
          f"{r['neg_rush']:>8.0f} {r['approx_tfl']:>10.0f} {r['tfl_actual']:>10.0f} "
          f"{r['tfl_gap']:>6.0f} {r['pct_captured']:>9.1f}%")

=== TFL back-calculation vs actual ===
Team            Seas  Sacks(plays)  NegRush  ApproxTFL  ActualTFL    Gap  %Captured
--------------------------------------------------------------------------------
  Alabama       2023            47       35         82         84      2      97.6%
  Georgia       2023            12       41         53         72     19      73.6%
  Iowa          2023            32       39         71         75      4      94.7%
  Michigan      2023            20       25         45         84     39      53.6%
  Ohio State    2023            15       25         40         70     30      57.1%
  Ole Miss      2023            23       61         84         82     -2     102.4%
  Penn State    2023            13       26         39        112     73      34.8%
  Texas         2023            26       33         59         79     20      74.7%
  Alabama       2024            22       56         78         71     -7     109.9%
  Georgia       2024            19      

In [59]:
# CELL 5 — KEEP — Rebuild Pool Assertion

cur.execute("""
    SELECT COUNT(*)
    FROM int.int_game_team_features f
    JOIN raw.games g ON g.id = f.game_id
    JOIN int.int_team_season_features ts
        ON ts.team_name = f.team_name AND ts.season = f.season
    JOIN int.int_team_season_features opp
        ON opp.season = f.season
        AND opp.team_name = CASE
            WHEN f.team_name = g.home_team THEN g.away_team
            ELSE g.home_team
        END
    WHERE g.season IN (2022, 2023, 2024, 2025)
      AND g.conference_game = TRUE
      AND g.week BETWEEN 5 AND 14
      AND ts.conference  != 'FBS Independents'
      AND opp.conference != 'FBS Independents'
      AND ts.conference  != 'Pac-12'
      AND opp.conference != 'Pac-12'
      AND NOT (ts.conference  = 'American Athletic' AND g.season = 2022)
      AND NOT (opp.conference = 'American Athletic' AND g.season = 2022)
      AND NOT (ts.conference  = 'Mountain West'     AND g.season = 2022)
      AND NOT (opp.conference = 'Mountain West'     AND g.season = 2022)
      AND NOT (ts.conference  = 'Big 12'            AND g.season = 2022)
      AND NOT (opp.conference = 'Big 12'            AND g.season = 2022)
""")
n_rows = cur.fetchone()[0]
assert n_rows == 3458, f"Expected 3,458 team-rows, got {n_rows}"
print(f"Team-rows in rebuild pool: {n_rows:,} — OK")

cur.execute("""
    SELECT COUNT(*)
    FROM int.int_team_season_features
    WHERE season IN (2022, 2023, 2024, 2025)
      AND conference != 'FBS Independents'
      AND conference != 'Pac-12'
      AND NOT (conference = 'American Athletic' AND season = 2022)
      AND NOT (conference = 'Mountain West'     AND season = 2022)
      AND NOT (conference = 'Big 12'            AND season = 2022)
""")
n_ts = cur.fetchone()[0]
assert n_ts == 457, f"Expected 457 team-seasons, got {n_ts}"
print(f"Team-seasons in rebuild pool: {n_ts:,} — OK")

print("\nRebuild pool assertions passed. Ready to build CTD features.")

Team-rows in rebuild pool: 3,458 — OK
Team-seasons in rebuild pool: 457 — OK

Rebuild pool assertions passed. Ready to build CTD features.
